# 🎭 **Masked Language Modeling with BERT**

Explore masked language modeling (MLM) using the BERT model to understand context and predict missing words in sentences.

## 🛠️ Setup and Installation

Begin by installing the necessary libraries to manage data processing and modeling.

In [ ]:
#!pip install -U transformers
#!pip3 install scipy


[notice] A new release of pip is available: 25.0 -> 25.3
[notice] To update, run: C:\Users\meisa\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## 📚 Importing Libraries

Import essential modules for our tasks.

In [1]:
from transformers import AutoModelForMaskedLM, AutoTokenizer
import pandas as pd
import numpy as np
#from scipy.special import softmax
#import tensorflow as tf

## 🤖 Model Setup

Load the pre-trained BERT model and tokenizer, specifically designed for masked language modeling.

In [3]:
model_name = "bert-base-cased"

# Loading the pre-trained model and tokenizer
model = AutoModelForMaskedLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

Some weights of the model checkpoint at bert-base-cased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


## 🎭 Defining the Mask Token

Identify the mask token used by BERT to signify where predictions are needed in the sentence.

## ✏️ Creating the Input Sentence

Craft a sentence with a missing word indicated by the mask token, to test the model's predictive power.

In [4]:
# Defining the mask token
mask = tokenizer.mask_token

# Defining the sentence
sentence = f"I want to {mask} pizza for tonight."

# Tokenizing the sentence
tokens = tokenizer.tokenize(sentence)
print(tokens)

['I', 'want', 'to', '[MASK]', 'pizza', 'for', 'tonight', '.']


## 🔍 Tokenization and Encoding

Tokenize and encode the sentence to format it properly for the model.

## 📈 Model Prediction

Feed the encoded inputs to the model and extract logits for predictions.

In [ ]:
# Encoding the input sentence and getting model predictions
encoded_inputs = tokenizer(sentence, return_tensors="pt")
output = model(**encoded_inputs)        # (batch_size, sequence_length, vocab_size)

# Detaching the logits from the model output and converting to numpy array
logits = output.logits.detach().numpy()[0]      

In [14]:
print(logits.shape)

(10, 28996)



## 🔎 Analyzing Predictions

Retrieve logits for the masked token and calculate confidence scores for possible replacements.

In [10]:
def softmax(x):
    x = x - np.max(x, axis=-1, keepdims=True)  # stability trick
    exp_x = np.exp(x)
    return exp_x / np.sum(exp_x, axis=-1, keepdims=True)

In [ ]:
# Extracting the logits for the masked token and calculating the confidence scores
masked_logits = logits[tokens.index(mask) + 1]      # tokens does not include [CLS]
confidence_scores = softmax(masked_logits)

In [15]:
print(confidence_scores.shape)

(28996,)


## 📝 Displaying Top Predictions

Cycle through the top 5 predicted tokens, substituting the masked token in the original sentence to show the model's suggestions.


In [ ]:
# Iterating over the top 5 predicted tokens and printing the sentences with the masked token replaced
for i in np.argsort(confidence_scores)[::-1][:5]:       # [::-1] reverse → descending
#for i in np.argsort(confidence_scores)[:5]:
    pred_token = tokenizer.decode(i)
    score = confidence_scores[i]

    # print(pred_token, score)
    print(sentence.replace(mask, pred_token))

I want to have pizza for tonight.
I want to get pizza for tonight.
I want to eat pizza for tonight.
I want to make pizza for tonight.
I want to order pizza for tonight.


: 